[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Engine and create_all &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the classes its worked examples wrote, `hero_engine`,
and `scratch/heroes.db` built and loaded as the notebook left it, Spider-Boy's missing team included.
Run it first. The tasks follow one another, and the last cell removes the scratch folder.


In [1]:
import contextlib
import logging
import re
import shutil
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlalchemy
import sqlmodel
from sqlalchemy import event, insert, inspect, text
from sqlalchemy.dialects import postgresql
from sqlalchemy.exc import IntegrityError
from sqlalchemy.schema import CreateTable
from sqlmodel import Field, SQLModel, create_engine

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


@contextlib.contextmanager
def catching():
    """Collects warnings instead of printing them: Python prints one with the file that raised it."""
    with warnings.catch_warnings(record=True) as raised:
        warnings.simplefilter("always")
        yield raised


class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from no database at all
Path("scratch").mkdir()

print("sqlmodel", sqlmodel.__version__, "| sqlalchemy", sqlalchemy.__version__, "| scratch:", Path("scratch").exists())

class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine

engine = create_engine("sqlite:///scratch/heroes.db")
SQLModel.metadata.create_all(engine)
with engine.begin() as connection:
    connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS[:2]])
    connection.execute(insert(Hero), [{"name": "Deadpond", "secret_name": "Dive Wilson", "team_id": 2},
                                      {"name": "Spider-Boy", "secret_name": "Pedro Parqueador", "team_id": 999}])


sqlmodel 0.0.42 | sqlalchemy 2.0.54 | scratch: True


**1.** An engine for a second database.


In [2]:
spare = create_engine("sqlite:///scratch/spare.db")
print("dialect :", spare.dialect.name, "| driver:", spare.dialect.driver)
print("database:", spare.url.database)
print("the file exists:", Path("scratch/spare.db").exists())

with spare.connect() as connection:
    pass
print("and after one connection:", Path("scratch/spare.db").exists())


dialect : sqlite | driver: pysqlite
database: scratch/spare.db
the file exists: False
and after one connection: True


The engine parsed the URL and left the disk alone; the file appeared when a connection asked for it.


**2.** The tables, in the second database.


In [3]:
SQLModel.metadata.create_all(spare)
print("tables:", inspect(spare).get_table_names())
for column in inspect(spare).get_columns("team"):
    print(f"  {column['name']:<14} {str(column['type']):<12} nullable {column['nullable']}")


tables: ['hero', 'team']
  id             INTEGER      nullable False
  name           VARCHAR(50)  nullable False
  headquarters   VARCHAR(60)  nullable False


`create_all` builds whatever the metadata holds on whichever engine it is given, so the same two
classes made the same two tables in a second file. `team`'s id is the primary key, which is never
null; the other two are `str`, so neither takes one either.


**3.** What it sends when there is nothing to do.


In [4]:
spare.echo = True
SQLModel.metadata.create_all(spare)
spare.echo = False
# Four: BEGIN, one PRAGMA table_info for each of the two tables, and COMMIT. No CREATE TABLE.


    BEGIN (implicit)
    PRAGMA main.table_info("team")
    PRAGMA main.table_info("hero")
    COMMIT


It asked the database about both tables, found them, and wrote nothing.


**4.** A third table, with a foreign key to the heroes.


In [5]:
class Sighting(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    hero_id: int | None = Field(default=None, foreign_key="hero.id")
    city: str = Field(max_length=60)


SQLModel.metadata.create_all(engine)
print("tables:", inspect(engine).get_table_names())
print("keys  :", [(key["constrained_columns"], key["referred_table"], key["referred_columns"])
                  for key in inspect(engine).get_foreign_keys("sighting")])


tables: ['hero', 'sighting', 'team']
keys  : [(['hero_id'], 'hero', ['id'])]


`create_all` created the one table that was missing and left `hero` and `team` alone.


**5.** The same sighting, on two engines.


In [6]:
with engine.begin() as connection:                                  # the engine with no connect event
    connection.execute(insert(Sighting), [{"hero_id": 999, "city": "Oslo"}])
with engine.connect() as connection:
    print("sightings written:", connection.execute(text("select hero_id, city from sighting")).all())

checked = hero_engine("scratch/heroes.db")
try:
    with checked.begin() as connection:
        connection.execute(insert(Sighting), [{"hero_id": 999, "city": "Bergen"}])
except IntegrityError as error:
    print(type(error).__name__ + ":", str(error).splitlines()[0])


sightings written: [(999, 'Oslo')]
IntegrityError: (sqlite3.IntegrityError) FOREIGN KEY constraint failed


Hero 999 does not exist, and the first engine wrote the row anyway. The second refused it, because
every connection it opens has the pragma on.


**6.** The same table, two databases.


In [7]:
for dialect in (engine.dialect, postgresql.dialect()):
    print(f"-- {dialect.name}")
    print(str(CreateTable(Sighting.__table__).compile(dialect=dialect)).strip())
# The id line: INTEGER on SQLite and SERIAL on PostgreSQL. And hero_id, which PostgreSQL writes as
# INTEGER too but SQLite spells the same way, so the id line is the only real difference here.


-- sqlite
CREATE TABLE sighting (
	id INTEGER NOT NULL, 
	hero_id INTEGER, 
	city VARCHAR(60) NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(hero_id) REFERENCES hero (id)
)
-- postgresql
CREATE TABLE sighting (
	id SERIAL NOT NULL, 
	hero_id INTEGER, 
	city VARCHAR(60) NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(hero_id) REFERENCES hero (id)
)


The `id` is the line that differs: SQLite numbers an `INTEGER PRIMARY KEY` itself, and PostgreSQL
needs `SERIAL` to say so. The foreign key and the `VARCHAR(60)` come out the same on both.

Last, the engines let go of their files, and this cell removes the scratch folder:


In [8]:
for made in (engine, spare, checked):
    made.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Engine and create_all](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/02-engine-and-create-all.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
